# Binary Classification of Pima-Indian Diabetes with PyTorch

Binary classification model using PyTorch, with [Pima Indians Diabetes dataset](https://www.openml.org/search?type=data&status=active&qualities.NumberOfClasses=%3D_2&id=46921)


The process will cover:
1. **Loading the Data**: Fetching the dataset from OpenML.
2. **Data Exploration & Visualization**: Understanding the features and their distributions.
3. **Data Preprocessing**: Splitting the data and scaling features.
4. **Model Building**: Creating a neural network with PyTorch.
5. **Training**: Training the model on our dataset.
6. **Evaluation**: Assessing the model's performance with various metrics.
7. **Saving the Model**: Persisting the trained model and scaler for future use.
8. **Inference**: Using the saved model to make predictions on new, unseen data.

In [ ]:
import openml
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Set plot style
sns.set(style="whitegrid")

## 1. Load the Dataset

We'll start by loading the `Pima Indians Diabetes dataset directly from OpenML using its dataset ID (37)`.

In [ ]:
dataset = openml.datasets.get_dataset(37)  # Pima-indians diabetes dataset
X, y, categorical_indicator, feature_names = dataset.get_data(
    dataset_format="dataframe", target=dataset.default_target_attribute
)

## 2. Data Exploration & Visualization

Before we start building the model, it's crucial to understand the data we're working with.
- `preg`: Number of times pregnant.
- ..
- `mass`: Body mass index
- `age`: Age (years)
- `class`: Target variable (0: tested -ve, 1: tested +ve for diabetes)

In [ ]:
print("Feature names:", feature_names)
print("\nFirst 5 rows of features (X):\n", X.head())

### Dataset Description

The dataset consists of several medical predictor variables and one target variable, `class`. The predictors include the number of pregnancies the patient has had, their BMI, insulin level, age, and so on.

In [ ]:
print(dataset.description)

### Target Variable Distribution

Let's see how many samples belong to each class. This helps us check for class imbalance.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x=y)
plt.title('Distribution of Diabetes Outcome')
plt.xlabel('Class (0: Negative, 1: Positive)')
plt.ylabel('Count')
plt.show()

### Feature Distributions

Histograms are a great way to visualize the distribution of each numerical feature.

In [ ]:
X.hist(figsize=(12, 10), bins=20)
plt.suptitle('Histograms of Input Features')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### Correlation Matrix

A heatmap of the correlation matrix helps us understand the relationships between different features.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(X.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Features')
plt.show()

## 3. Data Preprocessing

Now, we'll prepare the data for training. This involves:
1.  Converting the data to NumPy arrays with the correct data types.
2.  Splitting the data into training and testing sets.
3.  Scaling the input features to have zero mean and unit variance.

In [ ]:
# Convert to numpy / proper types
X_np = X.to_numpy().astype(np.float32)

# label encoding, mapping string labels to integers
y_np = y.map({"tested_negative": 0, "tested_positive": 1}).to_numpy().astype(np.int64) # Binary classification: Other encoding techniques can be used?

# Split into train / test
X_train, X_test, y_train, y_test = train_test_split(
    X_np, y_np, test_size=0.2, random_state=42, stratify=y_np
)

# Scale input features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")

## 4. Prepare Data for PyTorch

PyTorch models work with `Tensor` objects. We'll convert our NumPy arrays into Tensors and then create `DataLoader` objects to handle batching and shuffling.

In [ ]:
# Convert to torch tensors
X_train_t = torch.from_numpy(X_train_scaled)
y_train_t = torch.from_numpy(y_train)
X_test_t = torch.from_numpy(X_test_scaled)
y_test_t = torch.from_numpy(y_test)

# Create Dataset and DataLoader
train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

## 5. Define the Neural Network

We'll define a simple feed-forward neural network with three fully connected layers. The output layer uses a `Sigmoid` activation function to produce a probability between 0 and 1, which is suitable for binary classification.

In [ ]:
class DiabetesNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)  # output: single logit
        self.act = nn.ReLU()
        self.out_act = nn.Sigmoid()

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.out_act(self.fc3(x))
        return x

input_dim = X_train_t.shape[1]
model = DiabetesNet(input_dim)
print(model)

## 6. Train the Model

We'll use Binary Cross-Entropy Loss (`BCELoss`) since our output is a probability. The `Adam` optimizer is a popular and effective choice.

We'll train for a fixed number of epochs and record the loss at each epoch to visualize the training progress.

In [ ]:
criterion = nn.BCELoss()  # binary cross-entropy, since using Sigmoid
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 50
loss_history = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        y_pred = model(batch_X).squeeze(1)  # shape: (batch,)
        loss = criterion(y_pred, batch_y.float())
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_X.size(0)
    
    epoch_loss = running_loss / len(train_loader.dataset)
    loss_history.append(epoch_loss)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

### Visualize Training Loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs + 1), loss_history)
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

## 7. Evaluate the Model

After training, we evaluate the model's performance on the unseen test set. We'll look at:
- **Accuracy**: The proportion of correct predictions.
- **Confusion Matrix**: A table showing true positives, true negatives, false positives, and false negatives.
- **Classification Report**: Provides precision, recall, and F1-score for each class.
- **ROC Curve & AUC**: Measures the model's ability to distinguish between classes.

In [ ]:
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        y_pred_probs = model(batch_X).squeeze(1)
        all_preds.append(y_pred_probs.cpu().numpy())
        all_labels.append(batch_y.cpu().numpy())

all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

# Convert probabilities to binary labels (threshold = 0.5)
y_pred_labels = (all_preds >= 0.5).astype(int)

accuracy = accuracy_score(all_labels, y_pred_labels)
print(f"Test Accuracy: {accuracy:.4f}")

### Confusion Matrix and Classification Report

In [ ]:
cm = confusion_matrix(all_labels, y_pred_labels) # Confusion matrix shows the counts of true positive, true negative, false positive, false negative
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print("\nClassification Report:\n", classification_report(all_labels, y_pred_labels, target_names=['Negative', 'Positive']))

### ROC Curve and AUC

In [ ]:
fpr, tpr, thresholds = roc_curve(all_labels, all_preds)
auc = roc_auc_score(all_labels, all_preds) # Area Under Curve

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

## 8. Save Model and Scaler

To use the model for inference later without retraining, we save the model's learned parameters (`state_dict`) and the `StandardScaler` object.

In [ ]:
torch.save(model.state_dict(), "diabetes_model.pth")
joblib.dump(scaler, "scaler.pkl")
print("Model and scaler saved successfully.")

## 9. Inference on New Data

This final step shows how to `load the saved model and scaler` to make predictions on new, unseen data. 

This `simulates a real-world application` where you would deploy the model to predict outcomes for new patients.

In [ ]:
# This section demonstrates inference. Note that it re-loads the model and scaler we just saved.

# 1. Load the model and scaler
inference_model = DiabetesNet(input_dim)
inference_model.load_state_dict(torch.load("diabetes_model.pth"))
inference_model.eval()

inference_scaler = joblib.load("scaler.pkl")

# 2. Prepare new data (example rows)
# 'preg', 'plas', 'pres', 'skin', 'insu', 'mass', 'pedi', 'age'
X_new = np.array([[6, 148, 72, 35, 0, 33.6, 0.627, 50],  # Example from dataset
                  [1, 85, 66, 29, 0, 26.6, 0.351, 31]], dtype=np.float32) # Another example

# 3. Scale the new data
X_new_scaled = inference_scaler.transform(X_new)

# 4. Convert to a tensor
X_new_t = torch.from_numpy(X_new_scaled)

# 5. Make predictions
with torch.no_grad():
    y_new_pred_probs = inference_model(X_new_t).squeeze(1).cpu().numpy() # Probabilities

y_new_pred_labels = (y_new_pred_probs >= 0.5).astype(int) # Convert probabilities to binary labels

print("New data:")
print("\nFeature names:", feature_names)
print(X_new)
print("\nPredicted probabilities for new data:", y_new_pred_probs)
print("Predicted labels for new data:", y_new_pred_labels)

# Conclusion

- We have learned real world simulating concepts based on classification example.
- Data validation and analysis
- Saving `model and scaler`
- Loading `model and scaler`
- Inference testing